In [1]:
# 기본
import pandas as pd
import numpy as np

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split


# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

import os
import gc

In [33]:
#vif 하위20개 +segment만 있는 파일 생성
df2 = pd.read_csv('data/vif_step_21.csv',encoding='utf-8-sig')
df2=df2.drop(0)
check_col=set(list(df2['변수'].values)+['Segment'])

train_path='data/train/parquet'

df_list=[]
for x in os.listdir(train_path):
    file_path=os.path.join(train_path,x)
    
    df=pd.read_parquet(file_path)
    df=df[list(set(df.columns) & check_col|{'ID','기준년월'})]
    print(df.columns)
    
    
    gc.collect()
    
    df_list.append(df)
df=pd.merge(left=df_list[0],right=df_list[1],on=['ID','기준년월'])
for x in range(7):
    df=pd.merge(left=df,right=df_list[x+1],on=['ID','기준년월'])
    gc.collect()
#'ID','기준년월' 칼럼 제거
df=df.drop(['ID','기준년월'],axis=1)
df.to_csv('data/20칼럼segment.csv',encoding='utf-8-sig',index=False)

Index(['Segment', '이용금액_R3M_신용체크', 'ID', '_2순위카드이용금액', '기준년월'], dtype='object')
Index(['ID', '기준년월'], dtype='object')
Index(['쇼핑_도소매_이용금액', 'ID', '이용금액_오프라인_R6M', '_3순위쇼핑업종_이용금액', '_1순위업종_이용금액',
       '_2순위업종_이용금액', '정상입금원금_B5M', '연체입금원금_B0M', '_3순위업종_이용금액', '기준년월',
       '정상입금원금_B2M', '이용건수_신용_R12M', '최대이용금액_일시불_R12M', '_1순위교통업종_이용금액',
       '이용금액_오프라인_B0M', '이용건수_오프라인_R6M'],
      dtype='object')
Index(['청구금액_R6M', 'ID', '기준년월', '청구금액_B0'], dtype='object')
Index(['ID', '기준년월', '잔액_일시불_B0M', '평잔_일시불_3M'], dtype='object')
Index(['ID', '기준년월'], dtype='object')
Index(['ID', '기준년월'], dtype='object')
Index(['ID', '기준년월'], dtype='object')


In [48]:
#df= pd.read_csv('data/20칼럼segment.csv')
df2 = pd.read_csv('data/vif_step_11.csv',encoding='utf-8-sig')
#const 제거
df2=df2.drop(0)
check_col=set(list(df2['변수'].values)+['Segment'])
#원하는 칼럼만 선택
df=df[list(set(df.columns) & check_col)]


df1=df.copy()
# 입력과 결과로 나눈다.
X = df1.drop('Segment', axis=1)
print(X.columns,' 컬럼만 사용합니다')
y = df1['Segment']
# 문자열 -> 숫자

encoder1 = LabelEncoder()
encoder1.fit(y)
y2 = encoder1.transform(y)
# 입력 데이터 표준화
scaler1 = StandardScaler()
scaler1.fit(X)
X2 = scaler1.transform(X)
# 학습할 데이터를 변수에 담아준다.
# 학습용과 검증용으로 나눈다.
train_X, X_val, train_y, y_val = train_test_split(X2, y2, test_size=0.2, random_state=1)

Index(['_1순위교통업종_이용금액', '청구금액_B0', '연체입금원금_B0M', '잔액_일시불_B0M', '정상입금원금_B5M',
       '_2순위카드이용금액', '최대이용금액_일시불_R12M', '이용건수_신용_R12M', '_1순위업종_이용금액',
       '쇼핑_도소매_이용금액'],
      dtype='object')  컬럼만 사용합니다


In [33]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score
#학습
model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(encoder1.classes_),
    use_label_encoder=False,
    eval_metric='mlogloss'
)

model.fit(train_X, train_y)

print((y_val==model.predict(X_val)).mean())

0.85175


In [37]:
# 교차검증
f1_micro = make_scorer(f1_score, average='micro')
scores = cross_val_score(model, train_X, train_y, cv=5, scoring=f1_micro)
print("교차검증 평균 F1:", scores.mean())

교차검증 평균 F1: 0.4295111636380325


In [42]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300]
}

grid = GridSearchCV(
    estimator=XGBClassifier(tree_method="hist", device="cuda",objective='multi:softmax', num_class=5, use_label_encoder=False, eval_metric='mlogloss'),
    param_grid=param_grid,
    scoring='f1_micro',
    cv=3
)

grid.fit(train_X, train_y)

print("최적 F1:", grid.best_score_)
print("최적 파라미터:", grid.best_params_)

최적 F1: 0.8520064814814815
최적 파라미터: {'learning_rate': 0.3, 'max_depth': 7, 'n_estimators': 300}


In [50]:
best_model = XGBClassifier()
best_model.set_params(**grid.best_params_,tree_method="hist", device="cuda")
best_model.fit(train_X, train_y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.3, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [58]:
from sklearn.metrics import classification_report
y_val_pred = best_model.predict(X_val)
print("최종 F1 (micro):", f1_score(y_val, y_val_pred, average='micro'))
print(classification_report(y_val, y_val_pred, target_names=encoder1.classes_))

최종 F1 (micro): 0.8531458333333334
              precision    recall  f1-score   support

           A       0.52      0.14      0.22       203
           B       0.00      0.00      0.00        38
           C       0.64      0.44      0.52     25598
           D       0.57      0.41      0.48     69539
           E       0.90      0.96      0.93    384622

    accuracy                           0.85    480000
   macro avg       0.52      0.39      0.43    480000
weighted avg       0.84      0.85      0.84    480000

